# Notebook 6 – SQL

## `06_SQL.ipynb`

This notebook covers SQL from beginner to advanced level.

### Topics Covered
- Database Design
- Queries
- Joins
- Advanced SQL
- Window Functions

> **Note:** Examples use common SQL syntax. Some commands such as `LIMIT`, `FULL JOIN`, and certain database-specific features may vary between MySQL, PostgreSQL, SQL Server, and Oracle.

# 1. Database Design

Database design means organizing data into databases, tables, columns, relationships, and constraints so that data can be stored and retrieved efficiently.

We will use a simple **company database** for the examples.

## 1.1 Database

### Definition
A **database** is an organized collection of data that can be stored, managed, and retrieved using software called a Database Management System (DBMS).

### Example
A company database may contain:
- Employees
- Departments
- Projects
- Salaries

In [ ]:
-- MySQL / PostgreSQL example
CREATE DATABASE company_db;

-- Select the database in MySQL
USE company_db;

## 1.2 Tables

### Definition
A **table** stores data in rows and columns.

- **Column** = attribute/field
- **Row** = one record

Example: an `employees` table can contain employee ID, name, department, salary, and manager ID.

In [ ]:
CREATE TABLE departments (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(50)
);

CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(100),
    department_id INT,
    salary DECIMAL(10,2),
    manager_id INT
);

## 1.3 Primary Key

### Definition
A **Primary Key** uniquely identifies each row in a table.

### Rules
- Values must be unique.
- A primary key cannot contain `NULL`.
- A table normally has one primary key constraint.

Example: `employee_id` uniquely identifies an employee.

In [ ]:
CREATE TABLE students (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(100),
    age INT
);

## 1.4 Foreign Key

### Definition
A **Foreign Key** is a column that references a key in another table.

It helps maintain a relationship between tables and enforces referential integrity.

Example: `employees.department_id` references `departments.department_id`.

In [ ]:
CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(100),
    department_id INT,
    salary DECIMAL(10,2),
    FOREIGN KEY (department_id)
        REFERENCES departments(department_id)
);

## 1.5 Relationships

A relationship describes how tables are connected.

### Common relationship types

1. **One-to-One (1:1)** – one record is related to one record.
2. **One-to-Many (1:M)** – one department can have many employees.
3. **Many-to-Many (M:M)** – many students can take many courses. Usually implemented using a junction table.

In [ ]:
-- One-to-many example:
-- One department -> many employees

CREATE TABLE departments (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(50)
);

CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(100),
    department_id INT,
    FOREIGN KEY (department_id)
        REFERENCES departments(department_id)
);

## 1.6 Constraints

### Definition
Constraints are rules applied to table columns to control the type and validity of data.

### Common constraints
- `PRIMARY KEY`
- `FOREIGN KEY`
- `NOT NULL`
- `UNIQUE`
- `CHECK`
- `DEFAULT`

In [ ]:
CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(100) NOT NULL,
    email VARCHAR(100) UNIQUE,
    salary DECIMAL(10,2) CHECK (salary > 0),
    department_id INT,
    status VARCHAR(20) DEFAULT 'Active',
    FOREIGN KEY (department_id)
        REFERENCES departments(department_id)
);

# 3. SQL Queries

SQL queries are used to retrieve, filter, sort, group, and analyze data.

## 3.1 SELECT

### Definition
`SELECT` retrieves data from one or more tables.

### Syntax
```sql
SELECT column1, column2
FROM table_name;
```

In [ ]:
SELECT employee_id, employee_name, salary
FROM employees;

## 3.2 WHERE

### Definition
`WHERE` filters rows based on a condition.

In [ ]:
SELECT employee_name, salary
FROM employees
WHERE salary > 55000;

## 3.3 ORDER BY

### Definition
`ORDER BY` sorts query results.

- `ASC` = ascending (default)
- `DESC` = descending

In [ ]:
SELECT employee_name, salary
FROM employees
ORDER BY salary DESC;

## 3.4 GROUP BY

### Definition
`GROUP BY` groups rows with the same values and is commonly used with aggregate functions such as `COUNT()`, `SUM()`, `AVG()`, `MIN()`, and `MAX()`.

In [ ]:
SELECT department_id, COUNT(*) AS employee_count
FROM employees
GROUP BY department_id;

## 3.5 HAVING

### Definition
`HAVING` filters groups after `GROUP BY`.

### Important difference
- `WHERE` filters individual rows.
- `HAVING` filters grouped results.

In [ ]:
SELECT department_id, AVG(salary) AS average_salary
FROM employees
GROUP BY department_id
HAVING AVG(salary) > 50000;

## 3.6 DISTINCT

### Definition
`DISTINCT` removes duplicate values from the result.

In [ ]:
SELECT DISTINCT department_id
FROM employees;

## 3.7 LIMIT

### Definition
`LIMIT` restricts the number of rows returned.

> `LIMIT` is supported by MySQL and PostgreSQL. SQL Server commonly uses `TOP` or `OFFSET ... FETCH`, while Oracle has other row-limiting syntax.

In [ ]:
-- MySQL / PostgreSQL
SELECT employee_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 3;

# 4. SQL Joins

A **JOIN** combines rows from two or more tables using a related column.

Our examples connect `employees.department_id` with `departments.department_id`.

## 4.1 INNER JOIN

### Definition
Returns only rows that have matching values in both tables.

In [ ]:
SELECT
    e.employee_name,
    d.department_name
FROM employees e
INNER JOIN departments d
    ON e.department_id = d.department_id;

## 4.2 LEFT JOIN

### Definition
Returns all rows from the left table and matching rows from the right table.

If there is no match, columns from the right table contain `NULL`.

In [ ]:
SELECT
    e.employee_name,
    d.department_name
FROM employees e
LEFT JOIN departments d
    ON e.department_id = d.department_id;

## 4.3 RIGHT JOIN

### Definition
Returns all rows from the right table and matching rows from the left table.

> Some databases or versions may have limited support. A `LEFT JOIN` with the table order reversed can often achieve the same result.

In [ ]:
SELECT
    e.employee_name,
    d.department_name
FROM employees e
RIGHT JOIN departments d
    ON e.department_id = d.department_id;

## 4.4 FULL JOIN

### Definition
Returns all matching and non-matching rows from both tables.

> PostgreSQL supports `FULL OUTER JOIN`. MySQL does not directly support it; it can be simulated using `LEFT JOIN` + `RIGHT JOIN` with `UNION`.

In [ ]:
-- PostgreSQL / databases that support FULL OUTER JOIN
SELECT
    e.employee_name,
    d.department_name
FROM employees e
FULL OUTER JOIN departments d
    ON e.department_id = d.department_id;

## 4.5 CROSS JOIN

### Definition
A `CROSS JOIN` returns every possible combination of rows from the two tables.

If table A has 8 rows and table B has 4 rows, the result has 8 × 4 = 32 combinations.

In [ ]:
SELECT
    e.employee_name,
    d.department_name
FROM employees e
CROSS JOIN departments d;

# 5. Advanced SQL

## 5.1 UNION

### Definition
`UNION` combines the results of two compatible `SELECT` statements and removes duplicate rows.

The queries should have the same number of columns with compatible data types.

In [ ]:
SELECT employee_name AS name
FROM employees
WHERE department_id = 1

UNION

SELECT employee_name AS name
FROM employees
WHERE department_id = 4;

## 5.2 CASE

### Definition
`CASE` is used to perform conditional logic inside a SQL query.

It is similar to `if / elif / else` in programming.

In [ ]:
SELECT
    employee_name,
    salary,
    CASE
        WHEN salary >= 65000 THEN 'High Salary'
        WHEN salary >= 50000 THEN 'Medium Salary'
        ELSE 'Low Salary'
    END AS salary_category
FROM employees;

## 5.3 Subqueries

### Definition
A **subquery** is a query written inside another SQL query.

Example: find employees whose salary is greater than the average salary.

In [ ]:
SELECT employee_name, salary
FROM employees
WHERE salary > (
    SELECT AVG(salary)
    FROM employees
);

## 5.4 Common Table Expressions (CTEs)

### Definition
A **CTE** creates a temporary named result set that can be referenced by the main query.

CTEs improve readability and are useful for complex queries.

### Syntax
```sql
WITH cte_name AS (
    SELECT ...
)
SELECT ...
FROM cte_name;
```

In [ ]:
WITH high_salary_employees AS (
    SELECT employee_id, employee_name, salary
    FROM employees
    WHERE salary > 55000
)
SELECT *
FROM high_salary_employees;

## 5.5 Views

### Definition
A **View** is a virtual table based on the result of a SQL query.

Views can simplify frequently used queries and can help control which columns users see.

In [ ]:
CREATE VIEW employee_department_view AS
SELECT
    e.employee_id,
    e.employee_name,
    d.department_name,
    e.salary
FROM employees e
LEFT JOIN departments d
    ON e.department_id = d.department_id;

SELECT *
FROM employee_department_view;

## 5.6 Indexes

### Definition
An **index** is a database structure that can make data retrieval faster for suitable queries.

### Important point
Indexes can improve read/search performance, but they also require storage and can add overhead when inserting or updating data.

Do not create indexes on every column without considering query patterns.

In [ ]:
CREATE INDEX idx_employee_department
ON employees(department_id);

-- Example query that can benefit from an index on department_id
SELECT *
FROM employees
WHERE department_id = 1;

# 6. Window Functions

### Definition
A **window function** performs a calculation across related rows without collapsing the rows into one row per group.

Common syntax:

```sql
function_name() OVER (
    PARTITION BY column
    ORDER BY column
)
```

We will use:
- `ROW_NUMBER()`
- `RANK()`
- `DENSE_RANK()`
- `LAG()`
- `LEAD()`

## 6.1 ROW_NUMBER()

### Definition
Assigns a unique sequential number to each row within a result set or partition.

In [ ]:
SELECT
    employee_name,
    department_id,
    salary,
    ROW_NUMBER() OVER (
        PARTITION BY department_id
        ORDER BY salary DESC
    ) AS row_number
FROM employees;

## 6.2 RANK()

### Definition
Assigns the same rank to tied values. The next rank is skipped after a tie.

Example ranking: `1, 2, 2, 4`.

In [ ]:
SELECT
    employee_name,
    salary,
    RANK() OVER (
        ORDER BY salary DESC
    ) AS salary_rank
FROM employees;

## 6.3 DENSE_RANK()

### Definition
Assigns the same rank to tied values, but does not skip the next rank.

Example ranking: `1, 2, 2, 3`.

In [ ]:
SELECT
    employee_name,
    salary,
    DENSE_RANK() OVER (
        ORDER BY salary DESC
    ) AS salary_rank
FROM employees;

## 6.4 LAG()

### Definition
`LAG()` accesses a value from a previous row.

It is useful for comparing the current row with a previous row.

In [ ]:
SELECT
    employee_name,
    salary,
    LAG(salary) OVER (
        ORDER BY salary
    ) AS previous_salary
FROM employees;

## 6.5 LEAD()

### Definition
`LEAD()` accesses a value from a following row.

It is useful for comparing the current row with the next row.

In [ ]:
SELECT
    employee_name,
    salary,
    LEAD(salary) OVER (
        ORDER BY salary
    ) AS next_salary
FROM employees;

# 7. Window Functions – Department Example

Window functions become especially useful when combined with `PARTITION BY`.

The following query ranks employees separately inside each department.

In [ ]:
SELECT
    employee_name,
    department_id,
    salary,
    ROW_NUMBER() OVER (
        PARTITION BY department_id
        ORDER BY salary DESC
    ) AS row_number,
    RANK() OVER (
        PARTITION BY department_id
        ORDER BY salary DESC
    ) AS rank_number,
    DENSE_RANK() OVER (
        PARTITION BY department_id
        ORDER BY salary DESC
    ) AS dense_rank_number
FROM employees;

# 8. Practical SQL Examples

## Find the highest-paid employee

In [ ]:
SELECT employee_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 1;

## Find employees with their department names

In [ ]:
SELECT
    e.employee_name,
    d.department_name,
    e.salary
FROM employees e
LEFT JOIN departments d
    ON e.department_id = d.department_id
ORDER BY e.salary DESC;

## Find average salary by department

In [ ]:
SELECT
    department_id,
    AVG(salary) AS average_salary
FROM employees
GROUP BY department_id;

## Find departments having more than one employee

In [ ]:
SELECT
    department_id,
    COUNT(*) AS employee_count
FROM employees
GROUP BY department_id
HAVING COUNT(*) > 1;

## Find employees earning more than the company average

In [ ]:
SELECT employee_name, salary
FROM employees
WHERE salary > (
    SELECT AVG(salary)
    FROM employees
);

# 9. Quick Revision

| Topic | Purpose |
|---|---|
| Database | Organized collection of data |
| Table | Stores data in rows and columns |
| Primary Key | Uniquely identifies a row |
| Foreign Key | Connects related tables |
| Constraint | Enforces data rules |
| SELECT | Retrieves data |
| WHERE | Filters rows |
| ORDER BY | Sorts results |
| GROUP BY | Groups rows |
| HAVING | Filters groups |
| DISTINCT | Removes duplicates |
| LIMIT | Limits returned rows |
| INNER JOIN | Matching rows from both tables |
| LEFT JOIN | All left rows + matching right rows |
| RIGHT JOIN | All right rows + matching left rows |
| FULL JOIN | All rows from both sides |
| CROSS JOIN | Every possible row combination |
| UNION | Combines compatible query results |
| CASE | Conditional logic |
| Subquery | Query inside another query |
| CTE | Named temporary query result |
| View | Virtual table based on a query |
| Index | Helps speed suitable searches |
| ROW_NUMBER | Unique sequential row number |
| RANK | Ranking with gaps after ties |
| DENSE_RANK | Ranking without gaps after ties |
| LAG | Previous row value |
| LEAD | Next row value |

# Conclusion

You have now covered the major SQL topics required for beginner-to-intermediate database work:

**Database Design → Queries → Joins → Advanced SQL → Window Functions**

Practice each query by changing the conditions, columns, sorting order, and grouping columns. This will help you understand SQL instead of only memorizing syntax.